In [1]:
from sympy import (
    symbols, Function, sqrt, Rational, simplify,
    expand, collect, cosh, sinh, tanh, conjugate,
    Symbol, Add, Mul, Pow, srepr, latex, init_printing
)
from sympy import symbols, IndexedBase, Idx
import sympy as sp

init_printing(use_latex=True)

# Real parameters
r = symbols('r', real=True) # squeeze parameter

# k-dependent real coefficients (assume real for now)
eps, Delta, t_k, Lambda = symbols('epsilon Delta t Lambda', real=True)

# Shorthand for cosh/sinh of r
ch = cosh(r)
sh = sinh(r)

# Note: cosh²r - sinh²r = 1 will be used to simplify
print("Setup complete. ch² - sh² =", simplify(ch**2 - sh**2))

Setup complete. ch² - sh² = 1


In [10]:
# We represent each operator as a plain SymPy symbol.
# Naming convention:
# c_kA = c_{k,A} (annihilation, sublattice A, momentum k)
# c_kAd = c†_{k,A} (creation)
# c_mkA = c_{-k,A} (annihilation, momentum -k)
# c_mkAd = c†_{-k,A} (creation, momentum -k)
# Same pattern for sublattice B

# --- Original operators (before transformation) ---
c_kA, c_kAd = symbols(r'c_{kA} c^\dag_{kA}')
c_kB, c_kBd = symbols(r'c_{kB} c^\dag_{kB}')
c_mkA, c_mkAd = symbols(r'c_{-kA} c^\dag_{-kA}')
c_mkB, c_mkBd = symbols(r'c_{-kB} c^\dag_{-kB}')

# --- Transformed operators (after squeezing, denoted γ) ---
# These are what c_{k,σ} map TO under the squeeze
g_kA, g_kAd = symbols(r'γ_{kA} \gamma^\dag_{kA}')
g_kB, g_kBd = symbols(r'γ_{kB} \gamma^\dag_{kB}')
g_mkA, g_mkAd = symbols(r'γ_{-kA} \gamma^\dag_{-kA}')
g_mkB, g_mkBd = symbols(r'γ_{-kB} \gamma^\dag_{-kB}')

print("Operators defined.")

Operators defined.


In [16]:
# Squeeze transformation rules for each (k, σ) pair
# c_{k,σ} → ch * γ_{k,σ} + sh * γ†_{-k,σ}
# c†_{k,σ} → ch * γ†_{k,σ} + sh * γ_{-k,σ}

squeeze_rules = {
    # sublattice A, momentum +k
    c_kA: ch * g_kA + sh * g_mkAd,
    c_kAd: ch * g_kAd + sh * g_mkA,
    # sublattice A, momentum -k
    c_mkA: ch * g_mkA + sh * g_kAd,
    c_mkAd: ch * g_mkAd + sh * g_kA,
    # sublattice B, momentum +k
    c_kB: ch * g_kB + sh * g_mkBd,
    c_kBd: ch * g_kBd + sh * g_mkB,
    # sublattice B, momentum -k
    c_mkB: ch * g_mkB + sh * g_kBd,
    c_mkBd: ch * g_mkBd + sh * g_kB,
}

def squeeze(expr):
    """Apply squeeze transformation and expand."""
    return expand(expr.subs(squeeze_rules))

# --- Normal ordering engine ---
# Replace γ * γ† → γ†γ + 1 (bosonic: [γ, γ†] = 1)
# We track operator products as ordered pairs (op1, op2)
# and apply normal ordering to collect n̂ = γ†γ terms.

# Define normal-ordered number operators as new symbols
n_kA = symbols(r'n_{kA}') # γ†_{kA} γ_{kA}
n_kB = symbols(r'n_{kB}')
n_mkA = symbols(r'n_{-kA}')
n_mkB = symbols(r'n_{-kB}')

# Commutator substitution: γ γ† → γ†γ + 1
# Encoded as: symbol_pair → (normal_ordered_pair, +1 constant)
normal_order_rules = {
    g_kA * g_kAd: n_kA + 1,
    g_kB * g_kBd: n_kB + 1,
    g_mkA * g_mkAd: n_mkA + 1,
    g_mkB * g_mkBd: n_mkB + 1,
    # normal-ordered number ops are already fine
    g_kAd * g_kA: n_kA,
    g_kBd * g_kB: n_kB,
    g_mkAd * g_mkA: n_mkA,
    g_mkBd * g_mkB: n_mkB,
    g_kAd * g_kB: g_kAd * g_kB,   # hopping, normal ordered
    g_kBd * g_kA: g_kBd * g_kA,
}

def normal_order(expr):
    """Apply bosonic normal ordering [γ,γ†]=1."""
    return expand(expr.subs(normal_order_rules))

def transform(expr):
    """Full pipeline: squeeze → expand → normal order → simplify."""
    return simplify(normal_order(squeeze(expr)))

print("Transformation engine ready.")

Transformation engine ready.


In [17]:
# ============================================================
# TERM 1: ε (c†_{kA} c_{kA}) — diagonal kinetic, sublattice A
# ============================================================
T1_A = c_kAd * c_kA
T1_A_transformed = transform(T1_A)
print("T1_A (ε·c†kA·ckA) →", T1_A_transformed)
print()

# ============================================================
# TERM 2: -Δ/2 * c†_{kA} c†_{-kA} — pairing, sublattice A
# ============================================================
T2_A = c_kAd * c_mkAd
T2_A_transformed = transform(T2_A)
print("T2_A (-Δ/2·c†kA·c†-kA) →", T2_A_transformed)
print()

# h.c. of pairing term
T2_A_hc = c_mkA * c_kA
T2_A_hc_transformed = transform(T2_A_hc)
print("T2_A_hc (-Δ/2·c-kA·ckA) →", T2_A_hc_transformed)
print()

# ============================================================
# TERM 3: -t_k * c†_{kA} c_{kB} — inter-sublattice hopping
# ============================================================
T3 = c_kAd * c_kB
T3_transformed = transform(T3)
print("T3 (-t·c†kA·ckB) →", T3_transformed)
print()

# ============================================================
# TERM 4: -Λ_k * c†_{kA} c†_{-kB} — inter-sublattice pairing
# ============================================================
T4 = c_kAd * c_mkBd
T4_transformed = transform(T4)
print("T4 (-Λ·c†kA·c†-kB) →", T4_transformed)

T1_A (ε·c†kA·ckA) → \gamma^\dag_{-kA}*\gamma^\dag_{kA}*sinh(2*r)/2 + n_{-kA}*sinh(r)**2 + n_{kA}*cosh(r)**2 + γ_{-kA}*γ_{kA}*sinh(2*r)/2

T2_A (-Δ/2·c†kA·c†-kA) → \gamma^\dag_{-kA}*\gamma^\dag_{kA}*cosh(r)**2 + n_{-kA}*sinh(2*r)/2 + n_{kA}*sinh(2*r)/2 + γ_{-kA}*γ_{kA}*sinh(r)**2

T2_A_hc (-Δ/2·c-kA·ckA) → \gamma^\dag_{-kA}*\gamma^\dag_{kA}*sinh(r)**2 + n_{-kA}*sinh(2*r)/2 + n_{kA}*sinh(2*r)/2 + γ_{-kA}*γ_{kA}*cosh(r)**2

T3 (-t·c†kA·ckB) → \gamma^\dag_{-kB}*\gamma^\dag_{kA}*sinh(2*r)/2 + \gamma^\dag_{-kB}*γ_{-kA}*sinh(r)**2 + \gamma^\dag_{kA}*γ_{kB}*cosh(r)**2 + γ_{-kA}*γ_{kB}*sinh(2*r)/2

T4 (-Λ·c†kA·c†-kB) → \gamma^\dag_{-kB}*\gamma^\dag_{kA}*cosh(r)**2 + \gamma^\dag_{-kB}*γ_{-kA}*sinh(2*r)/2 + \gamma^\dag_{kA}*γ_{kB}*sinh(2*r)/2 + γ_{-kA}*γ_{kB}*sinh(r)**2


In [20]:
# ============================================================
# FULL HAMILTONIAN ASSEMBLY & TRANSFORMATION
# ============================================================

# --- Build the full Hamiltonian in terms of original operators ---
# H = Σ ε(c†kσ ckσ - Δ/2 c†kσ c†-kσ + h.c.)  [σ = A, B]
#   - t (c†kA ckB + h.c.)
#   - Λ (c†kA c†-kB + h.c.)

H = (# Inter-sublattice hopping (+ h.c.)
  - t_k * (c_kAd * c_kB + c_kBd * c_kA)
    # Inter-sublattice pairing (+ h.c.)
  - Lambda * (c_kAd * c_mkBd + c_mkB * c_kA)
)

# --- Apply squeeze transformation ---
H_transformed = transform(H)
print("Full transformed Hamiltonian:")
print(H_transformed)
print()

# --- Collect coefficients by operator type ---
# Define the operator basis we expect to appear
ops = [g_kAd*g_kBd, g_mkA*g_mkB,   # off-diag pairing (inter-sublattice)
       g_kAd*g_kB,  g_kBd*g_kA    # hopping terms
       ]

print("=" * 60)
print("Coefficients in the transformed Hamiltonian:")
print("=" * 60)

for op in ops:
    coeff = H_transformed.coeff(op)
    # coeff_simplified = simplify(coeff)
    if coeff != 0:
        print(f"\n  {op}  →  coeff = {coeff}")
        print(f"         LaTeX: {latex(coeff)}")

# Constant (vacuum energy) term — no operators
const_term = H_transformed
for op in ops:
    const_term = const_term - H_transformed.coeff(op) * op
const_term = simplify(expand(const_term))
print(f"\n  constant (vacuum energy)  →  {const_term}")
print(f"         LaTeX: {latex(const_term)}")

Full transformed Hamiltonian:
-Lambda*\gamma^\dag_{-kA}*\gamma^\dag_{kB}*sinh(r)**2 - Lambda*\gamma^\dag_{-kA}*γ_{-kB}*sinh(2*r)/2 - Lambda*\gamma^\dag_{-kB}*\gamma^\dag_{kA}*cosh(r)**2 - Lambda*\gamma^\dag_{-kB}*γ_{-kA}*sinh(2*r)/2 - Lambda*\gamma^\dag_{kA}*γ_{kB}*sinh(2*r)/2 - Lambda*\gamma^\dag_{kB}*γ_{kA}*sinh(2*r)/2 - Lambda*γ_{-kA}*γ_{kB}*sinh(r)**2 - Lambda*γ_{-kB}*γ_{kA}*cosh(r)**2 - \gamma^\dag_{-kA}*\gamma^\dag_{kB}*t*sinh(2*r)/2 - \gamma^\dag_{-kA}*t*γ_{-kB}*sinh(r)**2 - \gamma^\dag_{-kB}*\gamma^\dag_{kA}*t*sinh(2*r)/2 - \gamma^\dag_{-kB}*t*γ_{-kA}*sinh(r)**2 - \gamma^\dag_{kA}*t*γ_{kB}*cosh(r)**2 - \gamma^\dag_{kB}*t*γ_{kA}*cosh(r)**2 - t*γ_{-kA}*γ_{kB}*sinh(2*r)/2 - t*γ_{-kB}*γ_{kA}*sinh(2*r)/2

Coefficients in the transformed Hamiltonian:

  \gamma^\dag_{kA}*γ_{kB}  →  coeff = -Lambda*sinh(2*r)/2 - t*cosh(r)**2
         LaTeX: - \frac{\Lambda \sinh{\left(2 r \right)}}{2} - t \cosh^{2}{\left(r \right)}

  \gamma^\dag_{kB}*γ_{kA}  →  coeff = -Lambda*sinh(2*r)/2 - t*cosh(r)*